In this project, we are building a multi-agent system using the crewai library to analyze an image of a B2B digital menu startup. The system consists of four agents:

* Image Description Agent: Provides a detailed description of the image.
* UX Critique Agent: Critically analyzes the image based on the description.
* UX Suggestion Agent: Offers actionable suggestions to improve the image design.
* AI Product Manager: Writes user stories and prioritizes suggestions based on customer feedback.

# Setup

We start by setting up the working directory and installing the necessary libraries.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Change the working directory to the project folder in Google Drive
%cd /content/drive/MyDrive/Freelance/DataHeroes/Course/GenAI/GenAI/AI Agents/Capstone Project - the AI Product Manager

/content/drive/MyDrive/Freelance/DataHeroes/Course/GenAI/GenAI/AI Agents/Capstone Project - the AI Product Manager


In [3]:
# Install the 'crewai' library and its tools, along with 'openai'
!pip install 'crewai[tools]' langchain-openai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.2/197.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.4/836.4 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26

In [ ]:
!pip freeze

absl-py==1.4.0
accelerate==1.14.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.2
aiohttp==3.14.1
aiosignal==1.4.0
aiosqlite==0.21.0
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.12.0
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.14.0
anywidget==0.9.21
appdirs==1.4.4
apsw==3.53.2.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.1
astropy-iers-data==0.2026.6.22.1.23.34
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioread==3.1.0
Authlib==1.7.2
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
backoff==2.2.1
bcrypt==5.0.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.42.0
bigquery-magics==0.14.0
bleach==6.4.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.5.1
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.47
branca==0.8.2
brotli==1.2.0
build==1.5.1
CacheControl==0.14.4
cachetools==6.2

In [5]:
# Retrieve the OpenAI API key securely from Google Colab's user data
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

We are working in Google Colab and using Google Drive to store our project files. The crewai library is essential for creating and managing AI agents. We also retrieve the OpenAI API key securely to authenticate our API calls.

Next, we import all the necessary libraries for our project.

In [6]:
# Import essential libraries for image processing and agent creation
import os
from PIL import Image  # For image manipulation
from crewai_tools import VisionTool  # Specialized tool for image analysis
from crewai import Agent, Task, Crew, Process, LLM  # Core components of the crewai library
#from langchain_openai import ChatOpenAI  # Interface to interact with OpenAI's language models
from IPython.display import display, Markdown  # For displaying outputs in Jupyter notebooks

We import PIL for image handling, crewai components for agent management, and langchain_openai to interface with OpenAI's GPT-5 model. The VisionTool is a custom tool for image analysis.

In [7]:
# Set the OpenAI API key as an environment variable
os.environ['OPENAI_API_KEY'] = api_key

In [8]:
# Initialize the vision tool for image analysis
vision_tool = VisionTool()

By setting the API key as an environment variable, we ensure it's accessible throughout the program. The VisionTool will allow our agents to process and analyze images.

In [9]:
# Load the Image using the Pillow library
image_path = "translation.png"
image = Image.open(image_path)

The image translation.png is opened and stored in the image variable for later use.

# AGENT 1 - Image Description Agent

We create the first agent responsible for describing the image.

In [10]:
# Build the description agent with its role, goal, and backstory
description_agent = Agent(
    role="Image Description Agent",
    goal=f"Fully describe the digital image ({image_path}), of a B2B Digital Menu startup, including its visible elements, design, and intended purpose.",
    backstory="You are responsible for analyzing images and describing their purpose in detail.",
    verbose=True,
    tools=[vision_tool],
    llm=LLM(model="gpt-5.6-luna", temperature=0.8)
)


In [11]:
# Create the description task assigned to the description agent
description_task = Task(
    description="Identify and fully describe the digital image and explain its purpose.",
    expected_output="A complete description of the image and its purpose.",
    agent=description_agent
)

The Image Description Agent uses the VisionTool to analyze the image and provides a detailed description. We set the language model to GPT-5 with a temperature of 0.8 to allow for creative outputs.

# AGENT 2 - Critique Agent

We create the second agent to critique the image based on the description.

In [13]:
# Build the critique agent with its role, goal, and backstory
critique_agent = Agent(
    role="UX Critique Agent",
    goal=f"Critique the image {image_path} based on its description and intended purpose provided by the Image Description Agent.",
    backstory="You critically evaluate images, specifically UX designs, and point out flaws, weaknesses, and areas of improvement.",
    verbose=True,
    tools=[vision_tool],
    llm=LLM(model="gpt-5.6-luna", temperature=0.8)
)

In [14]:
# Create the critique task assigned to the critique agent, using the description task as context
critique_task = Task(
    description="Critically analyze the image based on its description and intended purpose.",
    expected_output="A complete critique of the image, highlighting design flaws and areas of improvement.",
    agent=critique_agent,
    context=[description_task]
)

The UX Critique Agent evaluates the image using the description from the first agent. The context parameter allows the agent to access the output of the description_task.

# AGENT 3 - UX Suggestion

We create the third agent to provide suggestions for improving the image.

In [15]:
# Create the UX suggestion agent with its role, goal, and backstory
ux_agent = Agent(
    role="UX Suggestion Agent",
    goal=f"Provide design and layout suggestions for the image {image_path} based on the context from the Image Description Agent and UX Critique Agent.",
    backstory="You specialize in providing actionable suggestions to improve the design of website images.",
    verbose=True,
    tools=[vision_tool],
    llm=LLM(model="gpt-5.6-luna", temperature=0.8)
)

In [16]:
# Create the UX suggestion task assigned to the UX suggestion agent, using previous tasks as context
ux_task = Task(
    description="Provide suggestions for improving the image design and layout, based on the context from the description and critique agents.",
    expected_output="A list of actionable suggestions for improving the image design and layout based on the image's purpose and critiques.",
    agent=ux_agent,
    context=[description_task, critique_task]
)

The UX Suggestion Agent leverages the outputs from both the description and critique agents to generate improvement suggestions.

# AGENT 4 - AI Product Manager

We create the fourth agent to write user stories and prioritize suggestions.

In [17]:
# Define the AI Product Manager Agent with its role, goal, and backstory
pm_agent = Agent(
    role="AI Product Manager",
    goal=f"Write user stories based on the suggestions from the UX agent for the image {image_path} and prioritize the suggestions based on probable customer feedback.",
    backstory="You act as a product manager for a digital company, prioritizing suggestions and creating user stories to guide improvements.",
    verbose=True,
    tools=[vision_tool],
    llm=LLM(model="gpt-5.6-luna", temperature=0.8)
)

In [18]:
# Create a task for the AI Product Manager, using all previous tasks as context
pm_task = Task(
    description="Write user stories based on the suggestions from the UX agent for the image and prioritize the suggestions based on probable customer feedback.",
    expected_output="A list of prioritized improvements based on expected impact on the customer, and the user stories.",
    agent=pm_agent,
    context=[description_task, critique_task, ux_task]
)

The AI Product Manager consolidates the suggestions and critiques to create actionable user stories, prioritizing them based on customer impact.

# Run the AI Product Manager

We now define the crew and run the tasks sequentially.

In [19]:
# Define the crew with all agents and tasks, set to run sequentially
crew = Crew(
    agents=[description_agent, critique_agent, ux_agent, pm_agent],
    tasks=[description_task, critique_task, ux_task, pm_task],
    verbose=True,
    process=Process.sequential
)

# Kick off the crew to start processing the tasks
result = await crew.kickoff_async()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: dfe60282-4dde-4513-8367-cb1236f83183                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Identify and fully describe the digital image and explain its purpose.                                   │
│  ID: 0c63c41f-6d1b-4b6e-80a6-a1825d4ba9e0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Image Description Agent                                                                                 │
│                                                                                                                 │
│  Task: Identify and fully describe the digital image and explain its purpose.                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: vision_tool                                                                                              │
│  Args: {'image_path_url': 'translation.png'}                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool vision_tool executed with result: The image shows a user interface for a menu translation center, possibly for a restaurant or food service application. It includes options for translating menu items from Portuguese to English. 

On t...


╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: vision_tool                                                                                              │
│  Output: The image shows a user interface for a menu translation center, possibly for a restaurant or food      │
│  service application. It includes options for translating menu items from Portuguese to English.                │
│                                                                                                                 │
│  On the left side, there are options related to managing the menu, such as categories and items. The main area  │
│  displays sections for "Comida" (Food), including "Entradas" (Appetizers) like "Bruschetta al Pomodoro" and     │
│  "Sopa de Mexilhão," as well as a section for "Pizzas" with options like "Margherita" and "Diavola." There are  │
│  fields for inputting translations in both languages, with "Pizzas" already filled in.                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (21 outputs left out)

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Product Manager                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Prioritized Improvements and User Stories                                                                    │
│                                                                                                                 │
│  Prioritization is based on probable customer impact, frequency of use, risk of lost work, and the effort       │
│  required for restaurant staff to translate and publish complete menus.                                         │
│                                                                                                                 │
│  ## Priority 1 — Show translation progress and incomplete content                                               │
│                                                                                                                 │
│  **Expected customer impact: Very high**                                                                        │
│  Restaurant managers need to know what remains untranslated before publishing. Without clear progress, they     │
│  may overlook individual dishes or categories.                                                                  │
│                                                                                                                 │
│  ### User stories                                                                                               │
│                                                                                                                 │
│  - **As a restaurant manager, I want to see the total number of translated and untranslated categories and      │
│  items so that I know whether the menu is complete.**                                                           │
│  - **As a restaurant manager, I want each category and menu item to display a completion status so that I can   │
│  quickly identify missing translations.**                                                                       │
│  - **As a restaurant manager, I want to filter the menu by status, such as Translated, Untranslated,            │
│  Incomplete, Needs review, and Published, so that I can focus on outstanding work.**                            │
│                                                                                                                 │
│  ### Acceptance criteria                                                                                        │
│                                                                                                                 │
│  - The page displays progress such as **“8 of 12 items translated.”**                                           │
│  - Category and item counts are shown separately.                                                               │
│  - Every category and item has a visible status label or icon.                                                  │
│  - Status is communicated through text or icons in addition to color.                                           │
│  - Users can filter the list by translation status.                                                             │
│  - A category is marked complete only when all required fields within it are complete.                          │
│                                                                                                                 │
│  ---                                                   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write user stories based on the suggestions from the UX agent for the image and prioritize the           │
│  suggestions based on probable customer feedback.                                                               │
│  Agent: AI Product Manager                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The Crew object manages all agents and tasks. By setting the process to Process.sequential, we ensure that each task is completed before the next one starts, maintaining the context flow.

Finally, we extract and display the outputs from each agent. We loop through each task's output and display it using Markdown formatting for better readability in the Jupyter notebook.

In [20]:
# extract and display the output of each agent
for idx, task_output in enumerate(result.tasks_output):
  display(Markdown(f"### Agent {idx+1}: {task_output.agent}\n{task_output.raw}"))


### Agent 1: Image Description Agent
The image is a desktop web interface for a B2B digital-menu management platform, specifically its menu-translation feature. It appears designed for restaurants or other food-service businesses that need to create, organize, and publish multilingual digital menus.

The interface has a clean, modern administrative-dashboard layout. A navigation panel on the left provides menu-management options, including categories and individual menu items. The main workspace is dedicated to translating menu content, with Portuguese and English used as the source and target languages.

The visible menu structure is organized under the Portuguese heading **“Comida”** (Food). Within it, there is an **“Entradas”** (Appetizers) category containing items such as:

- **Bruschetta al Pomodoro**
- **Sopa de Mexilhão**

A separate **“Pizzas”** category is also shown, with items including:

- **Margherita**
- **Diavola**

The central content area includes editable translation fields for category names and menu items. The Portuguese source text is displayed alongside or above fields where English translations can be entered. The **“Pizzas”** category appears to have an English translation already filled in, while other categories or items may still require translation.

The purpose of the image is to demonstrate a restaurant-facing translation workflow: a business can manage its food categories and dishes, enter equivalent names in another language, and prepare its digital menu for customers who speak different languages. This supports international guests, reduces manual translation work, and helps the restaurant maintain a consistent multilingual menu across its digital ordering or menu-display system.

### Agent 2: UX Critique Agent
## UX critique

The interface communicates its purpose reasonably well: it presents a restaurant menu hierarchy and provides Portuguese and English fields for translating categories and dishes. The left navigation also makes the translation feature discoverable within a broader menu-management system. However, the workflow appears more difficult and error-prone than necessary for restaurant staff, particularly when translating multiple items.

### Strengths

- **Clear overall information architecture:** The separation between “Restaurants,” “Edit Menu,” “Categories,” “Items,” and “Translation Center” gives users a recognizable administrative structure.
- **Menu hierarchy is visible:** Grouping items under “Comida,” “Entradas,” and “Pizzas” helps users understand where each translation belongs.
- **Source and target languages are identifiable:** Portuguese and English are presented as the translation direction, which reduces ambiguity.
- **Familiar dashboard layout:** A left sidebar and central workspace are conventional patterns for business-management software.
- **Existing translations can be distinguished:** The filled English translation for “Pizzas” suggests that users can see translation progress, at least implicitly.

### Key usability issues

#### 1. Translation progress is not sufficiently visible

The interface appears to show untranslated and translated content, but it does not clearly communicate the completion state of the menu. Users may not know:

- How many categories and items remain untranslated
- Which entries are incomplete
- Whether all required fields have been translated
- Whether translations are saved or merely entered
- Whether a category is complete while individual items are still missing

A visible progress indicator, such as “8 of 12 items translated,” completion badges, or color-coded status labels would make the workflow much easier to manage.

#### 2. The translation workflow appears fragmented

The description indicates that there are save buttons associated with individual language fields or sections. If users must save each item separately, translating a full menu could become repetitive and slow. It also increases the risk of:

- Forgetting to save a field
- Losing work when navigating to another item
- Creating inconsistent save behavior across categories
- Uncertainty about whether the current page contains unsaved changes

A better approach would be a single, clearly positioned “Save changes” action for the current set of edits, accompanied by autosave or an unsaved-changes warning.

#### 3. The source and target relationship could be clearer

The interface should make it immediately obvious which text is the original and which field is editable. If Portuguese and English are displayed in similar visual treatments, users may have to inspect labels carefully.

The source language should be explicitly marked as read-only, while the target field should be visually emphasized as editable. Clear labels such as:

- **Portuguese — original**
- **English — translation**

would prevent accidental editing of source content and reduce cognitive load.

#### 4. Navigation and editing context may be confusing

The left panel contains categories and items, while the central area also displays categories and items. This duplication may make it unclear whether the sidebar is for navigation, filtering, or editing.

The selected item or category should have a strong visual state, and the page should provide context such as:

> Translation Center / Comida / Entradas / Bruschetta al Pomodoro

A breadcrumb or persistent heading would help users understand exactly where they are in the menu hierarchy.

#### 5. The interface may not scale well for larger menus

The example contains only a few categories and dishes, but real restaurants may have dozens or hundreds of items. A vertically stacked layout with separate fields and save controls could become cumbersome.

Useful scalability features would include:

- Search by item name
- Filtering by translated, untranslated, or incomplete status
- Expand/collapse controls for categories
- Bulk translation actions
- Keyboard navigation between fields
- A table or spreadsheet-like editing mode
- Pagination or virtual scrolling for very large menus

#### 6. No indication of translation quality or review status

The system appears to treat translation as a simple text-entry task. That may be insufficient for food-service content, where terminology, allergens, ingredients, and cultural nuances matter.

The product could support review states such as:

- Draft
- Machine-translated
- Needs review
- Approved
- Published

It could also provide translation suggestions while keeping the restaurant owner in control. A “reviewed by” or last-updated indicator would be useful for teams.

#### 7. Save feedback is likely too weak

A save action needs immediate, unmistakable feedback. Without a success message, timestamp, or changed-state indicator, users may be unsure whether their translation was stored.

The interface should provide:

- A clear success confirmation
- Disabled save buttons when there are no changes
- Error messages near invalid or failed fields
- A visible “Unsaved changes” state
- Recovery from failed network requests

This is especially important in an administrative workflow where losing translations is costly.

#### 8. Language switching and directionality are underdeveloped

The example uses Portuguese and English, but the design should anticipate additional languages. The language selector should clearly support changing the target language without forcing users to leave the current menu.

If more languages are added, the layout must avoid becoming crowded. A language tab system, comparison view, or target-language dropdown would be more scalable than adding many columns or repeated forms.

#### 9. There is no apparent preview of the customer-facing result

Restaurant managers need to know how translations will appear to customers. A translation can be technically correct but visually too long, awkwardly truncated, or inconsistent with the published menu.

A preview toggle would help users inspect:

- The translated menu in its final layout
- Text wrapping and truncation
- Whether prices, descriptions, and allergens remain aligned
- How customers switch between languages

#### 10. Terminology and content consistency need support

The system should help maintain consistency across repeated words and categories. For example, “Entradas” could be translated as “Starters,” “Appetizers,” or “Small Plates,” and the chosen terminology should remain consistent throughout the menu.

Features such as a glossary, translation memory, or reusable category translations could reduce inconsistencies. The interface could also flag duplicate or conflicting translations.

### Visual and interaction improvements

- Increase contrast between read-only Portuguese text and editable English inputs.
- Use clear section headers and stronger grouping between categories and individual dishes.
- Make the active category or item unmistakable in the sidebar.
- Place the primary save action in a consistent, highly visible location.
- Avoid multiple similarly styled save buttons unless each one has a clearly defined scope.
- Add completion badges or icons beside categories and items.
- Use inline validation and concise helper text rather than relying on implicit behavior.
- Ensure form controls have sufficiently large click targets and visible keyboard-focus states.
- Provide responsive behavior for smaller screens, even if the primary use case is desktop.
- Use accessible labels instead of relying solely on placeholder text or visual positioning.
- Ensure status colors are not the only way to communicate completion or errors.

### Recommended improved workflow

1. Select a target language, such as English.
2. Display the menu hierarchy with completion status for every category and item.
3. Select a category or expand it to reveal its items.
4. Show the Portuguese source as read-only and the English field as editable.
5. Provide translation suggestions where appropriate.
6. Allow users to edit several entries without repeated navigation.
7. Save changes in one action, with autosave or an unsaved-changes warning.
8. Mark each entry as translated, reviewed, or published.
9. Offer a customer-facing preview before publishing.
10. Confirm that the translated version is live and indicate its publication status.

### Overall assessment

The design has a solid foundation and uses a familiar administrative-dashboard pattern, but it currently appears optimized for entering a few translations rather than efficiently managing a complete multilingual menu. The biggest opportunities are clearer progress tracking, more efficient bulk editing, stronger source/target distinction, reliable save feedback, and a preview-and-review workflow. These improvements would make the system more usable for restaurant teams and reduce mistakes before translated menus are published to customers.

### Agent 3: UX Suggestion Agent
- Add a prominent translation-progress indicator, such as **“8 of 12 items translated”**, with separate counts for categories and dishes.
- Show completion badges beside every category and item, using labels such as **Complete**, **Incomplete**, **Needs review**, and **Published**.
- Clearly distinguish the two languages with persistent labels: **Portuguese — Original** and **English — Translation**.
- Render Portuguese source text as read-only and use a stronger border, background, or focus treatment for editable English fields.
- Add a breadcrumb or contextual heading, for example: **Translation Center / Comida / Entradas / Bruschetta al Pomodoro**.
- Make the currently selected category or item unmistakable in the sidebar with a high-contrast active state.
- Clarify whether the left panel is used for navigation, filtering, or editing; avoid duplicating hierarchy without explaining its purpose.
- Replace multiple item-level save buttons with one consistent **Save changes** action for the current section or page.
- Add autosave or an **Unsaved changes** warning before users navigate away.
- Provide immediate save feedback, including success confirmations, timestamps, and clear error messages when saving fails.
- Disable the save button when no changes are pending, and show a visible loading state while changes are being submitted.
- Allow users to edit multiple translations in one view rather than forcing them to open and save each item individually.
- Add expand/collapse controls for categories such as **Entradas** and **Pizzas** to reduce scrolling on larger menus.
- Include search and filters for **translated**, **untranslated**, **incomplete**, **needs review**, and **published** entries.
- Add a bulk-edit or spreadsheet-style mode for restaurants with many menu items.
- Support keyboard navigation between translation fields to speed up repetitive entry.
- Add translation suggestions or machine-generated drafts, while keeping the restaurant manager responsible for approval.
- Introduce review states such as **Draft**, **Machine translated**, **Needs review**, **Approved**, and **Published**.
- Provide a glossary or translation-memory feature so recurring terms such as “Entradas” are translated consistently across the menu.
- Add helper text or validation for potentially inconsistent terminology, duplicate translations, and missing required fields.
- Make the target-language selector easy to find and allow users to switch languages without leaving the current menu location.
- Design the language selector to scale to multiple languages through tabs, a dropdown, or a comparison mode rather than adding excessive columns.
- Add a customer-facing preview that shows how the translated menu will look on the published digital menu.
- Include preview checks for text wrapping, truncation, category hierarchy, prices, allergens, and language-switching behavior.
- Use stronger visual grouping between category names and individual menu items through indentation, spacing, dividers, or nested cards.
- Improve contrast between labels, source text, input fields, and status indicators while maintaining a clean dashboard aesthetic.
- Ensure completion and error states are not communicated through color alone; pair colors with text, icons, or accessible status labels.
- Increase the size of form controls and action buttons to provide comfortable click targets for restaurant staff.
- Add visible keyboard-focus states and descriptive accessible labels for every input, selector, status, and action.
- Ensure the layout remains usable on smaller screens by allowing stacked language fields, collapsible navigation, and horizontally scrollable comparison areas where necessary.
- Add an explicit final workflow action such as **Review translations** or **Publish translated menu**, separate from saving drafts.
- Display the publication status and last-updated time so users know whether their changes are saved, approved, or already visible to customers.

### Agent 4: AI Product Manager
# Prioritized Improvements and User Stories

Prioritization is based on probable customer impact, frequency of use, risk of lost work, and the effort required for restaurant staff to translate and publish complete menus.

## Priority 1 — Show translation progress and incomplete content

**Expected customer impact: Very high**  
Restaurant managers need to know what remains untranslated before publishing. Without clear progress, they may overlook individual dishes or categories.

### User stories

- **As a restaurant manager, I want to see the total number of translated and untranslated categories and items so that I know whether the menu is complete.**
- **As a restaurant manager, I want each category and menu item to display a completion status so that I can quickly identify missing translations.**
- **As a restaurant manager, I want to filter the menu by status, such as Translated, Untranslated, Incomplete, Needs review, and Published, so that I can focus on outstanding work.**

### Acceptance criteria

- The page displays progress such as **“8 of 12 items translated.”**
- Category and item counts are shown separately.
- Every category and item has a visible status label or icon.
- Status is communicated through text or icons in addition to color.
- Users can filter the list by translation status.
- A category is marked complete only when all required fields within it are complete.

---

## Priority 2 — Make source and target languages unmistakable

**Expected customer impact: Very high**  
Confusion between the original Portuguese text and editable English fields could lead to accidental edits or incorrect translations.

### User stories

- **As a restaurant manager, I want persistent labels identifying Portuguese as the original language and English as the translation language so that I understand the translation direction immediately.**
- **As a restaurant manager, I want the original Portuguese content to be read-only so that I cannot accidentally change the source menu while translating.**
- **As a restaurant manager, I want editable English fields to have a distinct visual treatment so that I can immediately identify where to enter translations.**

### Acceptance criteria

- The interface clearly displays labels such as:
  - **Portuguese — Original**
  - **English — Translation**
- Portuguese fields cannot be edited from the translation screen.
- English fields have a clear input border, background, or other editable-state treatment.
- Labels remain visible while users scroll or edit.
- Screen readers can identify the source and target language for every field.

---

## Priority 3 — Replace fragmented saving with reliable save behavior

**Expected customer impact: Very high**  
Lost translations and uncertainty about whether changes were saved are major sources of frustration and operational risk.

### User stories

- **As a restaurant manager, I want to edit multiple translations and save them with one consistent action so that I do not have to save every item individually.**
- **As a restaurant manager, I want to know when I have unsaved changes so that I do not accidentally lose my work.**
- **As a restaurant manager, I want immediate confirmation after saving so that I know my translations were successfully stored.**
- **As a restaurant manager, I want a clear error message when saving fails so that I can retry without losing my changes.**

### Acceptance criteria

- A clearly positioned **Save changes** action saves all edits within the current page or section.
- The save button is disabled when no changes are pending.
- A loading state is shown while changes are submitted.
- Successful saves display a confirmation message and last-saved time.
- Failed saves show an actionable error message.
- Navigating away with unsaved changes triggers a warning.
- The system prevents accidental loss of entered translations during temporary network failures where technically feasible.

---

## Priority 4 — Support efficient multi-item translation

**Expected customer impact: High**  
Restaurant menus often contain dozens or hundreds of items. Translating one item at a time would be slow and repetitive.

### User stories

- **As a restaurant manager, I want to edit several category and item translations in one view so that I can translate a menu efficiently.**
- **As a restaurant manager, I want to expand or collapse categories such as Entradas and Pizzas so that I can control how much content is visible.**
- **As a restaurant manager, I want to move between translation fields using the keyboard so that I can enter repetitive translations faster.**
- **As a restaurant manager, I want a spreadsheet-style editing mode for large menus so that I can update many entries quickly.**

### Acceptance criteria

- Multiple translation fields can be edited without navigating to a new page.
- Categories can be expanded and collapsed independently.
- Keyboard navigation follows the menu hierarchy logically.
- The interface supports efficient editing of large menus without excessive scrolling.
- Bulk or spreadsheet editing is available as an optional mode for larger menus.

---

## Priority 5 — Improve navigation context and selection clarity

**Expected customer impact: High**  
The sidebar and main content both display menu structure, which may make it unclear where the user is working.

### User stories

- **As a restaurant manager, I want the selected category or item to be clearly highlighted so that I always know which content I am editing.**
- **As a restaurant manager, I want a breadcrumb showing my location in the menu hierarchy so that I understand the current editing context.**
- **As a restaurant manager, I want the sidebar’s purpose to be clear so that I know whether it is used for navigation, filtering, or editing.**

### Acceptance criteria

- The active sidebar item has a strong, accessible visual state.
- The page displays context such as:
  **Translation Center / Comida / Entradas / Bruschetta al Pomodoro**
- The selected sidebar item and main content remain synchronized.
- Navigation, filtering, and editing controls are visually differentiated.
- The interface does not duplicate menu hierarchy without explaining the relationship between the views.

---

## Priority 6 — Add customer-facing preview and publishing controls

**Expected customer impact: High**  
A translation may be correct but still appear poorly in the customer-facing menu due to truncation, wrapping, or layout issues.

### User stories

- **As a restaurant manager, I want to preview the translated menu as customers will see it so that I can identify presentation problems before publishing.**
- **As a restaurant manager, I want to verify text wrapping, prices, allergens, and category hierarchy so that the translated menu remains accurate and usable.**
- **As a restaurant manager, I want saving drafts to be separate from publishing so that I can review translations before customers see them.**
- **As a restaurant manager, I want to see whether translations are saved, approved, or published so that I understand their current availability to customers.**

### Acceptance criteria

- A **Preview** action is available from the translation workflow.
- Preview reflects the customer-facing menu layout.
- Preview supports switching between available languages.
- The system highlights or makes visible truncation, overflow, and layout issues.
- **Save draft**, **Review translations**, and **Publish translated menu** are separate actions.
- Publication status and last-updated time are visible.

---

## Priority 7 — Add review states and translation assistance

**Expected customer impact: Medium to high**  
Restaurants may need to review machine-generated or staff-entered translations, especially for ingredients, allergens, and culturally specific food terminology.

### User stories

- **As a restaurant manager, I want translation suggestions so that I can create translations faster while retaining control over the final wording.**
- **As a reviewer, I want to mark a translation as Draft, Needs review, Approved, or Published so that my team knows its current state.**
- **As a restaurant manager, I want to see who reviewed a translation and when so that accountability is maintained.**
- **As a restaurant manager, I want to approve or edit suggested translations before publication so that customers receive accurate information.**

### Acceptance criteria

- The system may provide machine-generated or suggested translations.
- Suggested translations are clearly identified as suggestions.
- Users can edit suggestions before saving.
- Each translation supports review states:
  - Draft
  - Machine translated
  - Needs review
  - Approved
  - Published
- The system records reviewer and last-updated information.
- No suggested translation is automatically published without the configured approval step.

---

## Priority 8 — Support terminology consistency with a glossary

**Expected customer impact: Medium**  
Inconsistent translations of repeated categories or ingredients can make the menu appear unprofessional and confuse customers.

### User stories

- **As a restaurant manager, I want to define preferred translations for recurring terms so that the menu uses consistent terminology.**
- **As a restaurant manager, I want the system to reuse approved translations for repeated terms so that I do not have to translate the same phrase multiple times.**
- **As a reviewer, I want the system to flag conflicting translations so that I can resolve inconsistencies before publishing.**

### Acceptance criteria

- Users can create glossary entries for recurring menu terms.
- Approved glossary translations can be suggested automatically.
- The system identifies inconsistent translations of the same source term.
- Users can choose whether to apply a glossary translation globally or only to selected entries.
- Glossary changes do not silently overwrite existing approved translations.

---

## Priority 9 — Improve language selection and support additional languages

**Expected customer impact: Medium**  
The current Portuguese-to-English workflow should scale as restaurants add Spanish, French, German, or other languages.

### User stories

- **As a restaurant manager, I want to select the target language without leaving the current menu location so that I can translate multiple languages efficiently.**
- **As a restaurant manager, I want the interface to support multiple target languages without becoming visually crowded.**
- **As a restaurant manager, I want to compare source and target languages in a consistent layout so that I can maintain translation accuracy.**

### Acceptance criteria

- The target-language selector is easy to find.
- Changing the target language preserves the current category or item context.
- The interface supports additional languages through tabs, a dropdown, or comparison mode.
- Users can see which languages have complete, incomplete, or published translations.
- Adding another language does not require excessive horizontal scrolling or duplicated controls.

---

## Priority 10 — Improve visual grouping, accessibility, and responsive behavior

**Expected customer impact: Medium**  
These improvements reduce daily friction and make the workflow usable for more staff members and devices.

### User stories

- **As a restaurant employee, I want category names and menu items to be visually grouped so that I can understand the menu hierarchy quickly.**
- **As a keyboard user, I want visible focus states and logical tab order so that I can complete translations without a mouse.**
- **As a user with visual or color-vision limitations, I want statuses and errors communicated through text and icons as well as color so that I can understand the interface reliably.**
- **As a restaurant manager using a smaller screen, I want the translation interface to remain usable so that I can manage the menu from different devices.**

### Acceptance criteria

- Categories and items use indentation, spacing, dividers, or nested cards to show hierarchy.
- All controls have descriptive accessible labels.
- Keyboard focus is clearly visible.
- Interactive controls have sufficiently large click targets.
- Statuses and errors are not communicated by color alone.
- The layout supports stacked source and target fields on smaller screens.
- The sidebar can collapse or become a mobile navigation control.
- Language comparison areas can scroll horizontally when necessary without breaking the page layout.

# Recommended Delivery Order

1. Translation progress and completion statuses  
2. Clear source/target language distinction  
3. Reliable unified save behavior and unsaved-change protection  
4. Multi-item editing, category expansion, and keyboard navigation  
5. Navigation context and selected-item clarity  
6. Customer preview and separate review/publish workflow  
7. Review states and translation suggestions  
8. Glossary and translation consistency tools  
9. Scalable multi-language selection  
10. Accessibility, visual polish, and responsive improvements